<a href="https://colab.research.google.com/github/author-sanjay/AirSafetyAI/blob/Data-Normalization/AirCraftAccidentDataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import requests
from bs4 import BeautifulSoup
import time
import csv
from google.colab import drive
from collections import defaultdict
import json
import pandas as pd
import re
import numpy as np
from datetime import datetime

# Data Collection

###### Data collection has been done already from airsafety db from year 2000 to 2025 resulting up to 6500+ recorded incidents found to train the model. Please note that this data is only being used for research purposes and model training

# Data Processing

#### Finding and Deleting Duplicates in dat

In [34]:


file_path = "/content/drive/MyDrive/accidents.json"

# Load your data
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Dictionary to track occurrences
seen = defaultdict(list)

for idx, entry in enumerate(data):
    # Composite key: Date + Time + Registration + Location
    key = f"{entry.get('Date','')}_{entry.get('Time','')}_{entry.get('Registration','')}_{entry.get('Location','')}"
    seen[key].append(idx)

# Find duplicates (keys with more than 1 entry)
duplicates = {k: v for k, v in seen.items() if len(v) > 1}

print(f"✅ Total entries: {len(data)}")
print(f"⚠️ Potential duplicates found: {len(duplicates)}")

✅ Total entries: 6791
⚠️ Potential duplicates found: 0


#### Data Normalisation

In [35]:


file_path = "/content/drive/MyDrive/accidents.json"
cleaned_file_path = "/content/drive/MyDrive/accidents_cleaned.json"
csv_file_path = "/content/drive/MyDrive/accidents_cleaned.csv"

# --- Load JSON into pandas ---
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)


###### Fixing Unknown Dates

In [36]:


# --- Handle unk. date YYYY ---
def normalize_date(val):
    if pd.isna(val):
        return None
    val = str(val).strip()

    # If format is "unk. date YYYY"
    match = re.match(r"unk\. date (\d{4})", val, flags=re.IGNORECASE)
    if match:
        year = int(match.group(1))
        return f"{year}-12-31"

    # Try normal parsing
    try:
        return pd.to_datetime(val, errors="coerce").strftime("%Y-%m-%d")
    except Exception:
        return None

df["Date"] = df["Date"].apply(normalize_date)
print(df.head())


         Date      Time                            Type  \
0  2000-01-01  13:00 LT          Cessna 550 Citation II   
1  2000-01-03             Beechcraft 200 Super King Air   
2  2000-01-04  17:25 LT  Beechcraft B200 Super King Air   
3  2000-01-05     13:25  Embraer EMB-110P1A Bandeirante   
4  2000-01-07                             Antonov An-26   

                    Owner/operator Registration       MSN Total airframe hrs  \
0               US Customs Service       N752CC  550-0018        12159 hours   
1  Kalahari Air Services & Charter       A2-AEZ    BB-421                NaN   
2                          Private       N895TT   BB-1239         3238 hours   
3         Skypower Express Airways       5N-AXL    110455                NaN   
4                          Unknown       D2-FBR      7206                NaN   

                     Engine model                     Fatalities  \
0                     P&W JT15D-4   Fatalities: 0 / Occupants: 3   
1                           

Normalizing Time

In [37]:
def normalize_time(val):
    val = str(val).strip()
    if not val:
        # missing or empty → set to noon
        return "12:00"

    # Some have "LT" suffix
    val = val.replace("LT", "").strip()

    try:
        t = pd.to_datetime(val, format="%H:%M", errors="coerce")
        if pd.isna(t):
            return "12:00"   # fallback if weird format
        return t.strftime("%H:%M")
    except:
        return "12:00"

df["Time"] = df["Time"].apply(normalize_time)
print(df.head())

         Date   Time                            Type  \
0  2000-01-01  13:00          Cessna 550 Citation II   
1  2000-01-03  12:00   Beechcraft 200 Super King Air   
2  2000-01-04  17:25  Beechcraft B200 Super King Air   
3  2000-01-05  13:25  Embraer EMB-110P1A Bandeirante   
4  2000-01-07  12:00                   Antonov An-26   

                    Owner/operator Registration       MSN Total airframe hrs  \
0               US Customs Service       N752CC  550-0018        12159 hours   
1  Kalahari Air Services & Charter       A2-AEZ    BB-421                NaN   
2                          Private       N895TT   BB-1239         3238 hours   
3         Skypower Express Airways       5N-AXL    110455                NaN   
4                          Unknown       D2-FBR      7206                NaN   

                     Engine model                     Fatalities  \
0                     P&W JT15D-4   Fatalities: 0 / Occupants: 3   
1                             NaN      Fatalit

Merging Time and Date into one column

In [38]:
df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    errors="coerce"   # in case something slips through
)

# Step 4: drop old columns
df.drop(columns=["Date", "Time"], inplace=True)

Convert to UTC

In [39]:
df["Datetime"] = df["Datetime"].dt.tz_localize("UTC")

Normalizinng Categories

In [40]:
print(df["Category"].unique())

['Accident' 'UK' 'Other' 'Unlawful Interference' nan 'Incident'
 'Serious incident']


In [41]:
import numpy as np

def normalize_category(cat):
    if pd.isna(cat):  # handle NaN
        return "Incident"   # fallback instead of dropping
    cat = str(cat).strip().lower()
    if cat == "accident":
        return "Accident"
    elif cat == "incident":
        return "Incident"
    elif cat == "serious incident":
        return "Serious Incident"
    elif cat == "unlawful interference":
        return "Unlawful Interference"
    elif cat == "uk":
        return "Unknown"
    elif cat == "other":
        return "Other"
    else:
        return "Incident"  # fallback if anything weird pops up

df["Category"] = df["Category"].apply(normalize_category)
print(df["Category"].unique())

['Accident' 'Unknown' 'Other' 'Unlawful Interference' 'Incident'
 'Serious Incident']


Dropping Operator Adds no Prediction value

In [42]:
df.drop(columns=["Owner/operator"], inplace=True)

Dropping MSN And Registration. Add no value to

In [43]:
df.drop(columns=["Registration", "MSN"], inplace=True)

Normalizing Flight Hours

In [44]:
def clean_hours(x):
    if pd.isna(x) or str(x).strip() == "":
        return np.nan
    try:
        return float(str(x).split()[0].replace(",", ""))  # e.g. "12159 hours" → 12159
    except:
        return np.nan

df["Total airframe hrs"] = df["Total airframe hrs"].apply(clean_hours)

# --- Step 2: Bucketize hours ---
def bucketize_hours(x):
    if pd.isna(x):
        return "Mid-life"   # Missing treated as mid-life
    elif x < 5000:
        return "New"
    elif x < 20000:
        return "Mid-life"
    else:
        return "Old"

df["airframe_bucket"] = df["Total airframe hrs"].apply(bucketize_hours)

# --- Step 3: Aircraft Age
df["Year of manufacture"] = pd.to_numeric(df["Year of manufacture"], errors="coerce")

# Aircraft age (years)
df["aircraft_age"] = df["Datetime"].dt.year - df["Year of manufacture"]


# --- Step 4: Handle NaNs ---
df.loc[df["Year of manufacture"].isna(), "aircraft_age"] = np.nan

In [45]:
print(df.head())

                             Type  Total airframe hrs  \
0          Cessna 550 Citation II             12159.0   
1   Beechcraft 200 Super King Air                 NaN   
2  Beechcraft B200 Super King Air              3238.0   
3  Embraer EMB-110P1A Bandeirante                 NaN   
4                   Antonov An-26                 NaN   

                     Engine model                     Fatalities  \
0                     P&W JT15D-4   Fatalities: 0 / Occupants: 3   
1                             NaN      Fatalities:  / Occupants:   
2                   P&W PT-6-A-42   Fatalities: 0 / Occupants: 3   
3  Pratt & Whitney Canada PT6A-34  Fatalities: 1 / Occupants: 13   
4                  Ivchenko AI-24    Fatalities:  / Occupants: 8   

  Other fatalities                Aircraft damage  Category  \
0                0                    Substantial  Accident   
1                0         Destroyed, written off   Unknown   
2                0                    Substantial  Accident

Dropping DateTime, Not Needed now.

In [46]:
df.drop(columns=["Datetime"], inplace=True)

Dropping Investigating Agence

In [47]:
df.drop(columns=["Investigating agency"], inplace=True)

Airports not needed

In [48]:
df.drop(columns=["Destination airport","Departure airport"], inplace=True)

Don't need confidence rating Or Location

In [49]:
df.drop(columns=["Confidence Rating","Location"], inplace=True)

We don't need Fatalities, as it will add bias in model considering more deadly accident as more serious. Thats not what we want

In [50]:
df.drop(columns=["Fatalities","Other fatalities"], inplace=True)

Normalise Phase

In [51]:
phase_mapping = {
    "Approach": "Approach",
    "Landing": "Landing",
    "En route": "Cruise",
    "Taxi": "Ground",
    "Take off": "Takeoff",
    "Initial climb": "Takeoff",
    "Pushback / towing": "Ground",
    "Standing": "Ground",
    "Manoeuvring  (airshow, firefighting, ag.ops.)": "Special ops",
    "Unknown": "Unknown",
    "": "Unknown"
}

df["Phase"] = df["Phase"].map(lambda x: phase_mapping.get(str(x).strip(), "Unknown"))


Remove "Ground" Phase because model is for inflight emergancies

In [52]:
df = df[df["Phase"] != "Ground"].reset_index(drop=True)


Normalizing Nature

In [53]:
nature_mapping = {
    "Passenger - Scheduled": "Passenger",
    "Passenger - Non-Scheduled/charter/Air Taxi": "Passenger",
    "Passenger": "Passenger",
    "Executive": "Passenger",
    "Cargo": "Cargo",
    "Military": "Military",
    "Private": "Private",
    "Training": "Training",
    "Ferry/positioning": "Ferry/Positioning",
    "Fire fighting": "Special Operations",
    "Parachuting": "Special Operations",
    "Agricultural": "Special Operations",
    "Ambulance": "Special Operations",
    "Survey": "Special Operations",
    "Aerial patrol": "Special Operations",
    "Test": "Test/Demo",
    "Calibration/Inspection": "Test/Demo",
    "Demo/Airshow/Display": "Test/Demo",
    "Illegal Flight": "Illegal",
    "Unknown": "Unknown",
    "": "Unknown",
    "-": "Unknown",
    "SF": "Unknown"
}

df["Nature"] = df["Nature"].replace(nature_mapping)
print(df["Nature"].unique())


['Ferry/Positioning' 'Unknown' 'Private' 'Passenger' 'Cargo' 'Training'
 'Military' 'Special Operations' 'Illegal' 'Test/Demo']


In [54]:
print(df["Nature"].unique())

['Ferry/Positioning' 'Unknown' 'Private' 'Passenger' 'Cargo' 'Training'
 'Military' 'Special Operations' 'Illegal' 'Test/Demo']
